## Setup and imports

In [ ]:
EXP_NAME = "25_MAY_class_B"

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
  from google.colab import userdata
  from google.colab import drive
  drive.mount('/content/drive')
  PROJECT_ROOT = userdata.get('PROJECT_ROOT')
else:
  PROJECT_ROOT = '../..'
  if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_rel
from src.config import Config

sns.set_style('whitegrid')
sns.set_context('paper', font_scale=1)

## Function library

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, roc_auc_score
from scipy.stats import sem, t

def get_95_ci(data):
  '''
    Calculates the 95% confidence interval for a given set of data.

    Inputs
      data: data as a Pandas Series

    Outputs
      interval: Array of the lower and upper bounds of the confidence interval
  '''
  n = len(data)
  mean = data.mean()
  std_err = sem(data, nan_policy='omit')
  if mean == 0 and std_err == 0:
    return (0.0, 0.0)
  interval = t.interval(0.95, n - 1, loc=mean, scale=std_err)
  return interval

def format_avg(metric, precision=3, perc=False):
  ci = get_95_ci(metric)
  if perc:
    return f"{round(metric.mean()*100, precision)}% ({round(ci[0]*100, precision)}%-{round(ci[1]*100, precision)}%)"
  else:
    return f"{round(metric.mean(), precision)} ({round(ci[0], precision)}-{round(ci[1], precision)})"

In [ ]:

baseline_metrics = pd.read_csv(f'{PROJECT_ROOT}/{Config.RESULTS_DIR}/{EXP_NAME}/baseline_metrics.csv')
ablation_metrics = pd.read_csv(f'{PROJECT_ROOT}/{Config.RESULTS_DIR}/{EXP_NAME}/ablation_metrics.csv')
baseline_unaware_metrics = pd.read_csv(f'{PROJECT_ROOT}/{Config.RESULTS_DIR}/{EXP_NAME}/baseline_unaware_metrics.csv')
fair_metrics = pd.read_csv(f'{PROJECT_ROOT}/{Config.RESULTS_DIR}/{EXP_NAME}/fair_metrics.csv')
fair_unaware_metrics = pd.read_csv(f'{PROJECT_ROOT}/{Config.RESULTS_DIR}/{EXP_NAME}/fair_unaware_metrics.csv')

baseline_feat_imp = pd.read_csv(f'{PROJECT_ROOT}/{Config.RESULTS_DIR}/{EXP_NAME}/baseline_feat_imp.csv')
ablation_feat_imp = pd.read_csv(f'{PROJECT_ROOT}/{Config.RESULTS_DIR}/{EXP_NAME}/ablation_feat_imp.csv')
baseline_unaware_feat_imp = pd.read_csv(f'{PROJECT_ROOT}/{Config.RESULTS_DIR}/{EXP_NAME}/baseline_unaware_feat_imp.csv')
fair_feat_imp = pd.read_csv(f'{PROJECT_ROOT}/{Config.RESULTS_DIR}/{EXP_NAME}/fair_feat_imp.csv')
fair_unaware_feat_imp = pd.read_csv(f'{PROJECT_ROOT}/{Config.RESULTS_DIR}/{EXP_NAME}/fair_unaware_feat_imp.csv')

## Performance and group fairness

In [ ]:
print(f"Classification threshold: {baseline_metrics['threshold'].mean().round(3)}\n")

perf_summary = pd.DataFrame({
'Model': [
        'Baseline: Xdesc, Xcorr, Xind, Sens', 
        'Baseline S-unaware: Xdesc, Xcorr, Xind',
        'Ablation: Xcorr, Xind',
        'Fair: Udesc, Xcorr, Xind, Sens',
        'Fair S-unaware: Udesc, Xcorr, Xind'],
# 'Class. Threshold': [
#         baseline_metrics['threshold'].mean().round(3),
#         baseline_unaware_metrics['threshold'].mean().round(3),
#         fair_metrics['threshold'].mean().round(3),
#         fair_unaware_metrics['threshold'].mean().round(3)
# ],
'Accuracy (95% CI)': [
        format_avg(baseline_metrics['accuracy']),
        format_avg(baseline_unaware_metrics['accuracy']),
        format_avg(ablation_metrics['accuracy']),
        format_avg(fair_metrics['accuracy']),
        format_avg(fair_unaware_metrics['accuracy'])
        ],
'Brier score (95% CI)': [
        format_avg(baseline_metrics['brier_score']),
        format_avg(baseline_unaware_metrics['brier_score']),
        format_avg(ablation_metrics['brier_score']),
        format_avg(fair_metrics['brier_score']),
        format_avg(fair_unaware_metrics['brier_score'])
        ],
'AUPRC (95% CI)': [
        format_avg(baseline_metrics['auprc']),
        format_avg(baseline_unaware_metrics['auprc']),
        format_avg(ablation_metrics['auprc']),
        format_avg(fair_metrics['auprc']),
        format_avg(fair_unaware_metrics['auprc'])
        ],
'FNR (95% CI)': [
        format_avg(baseline_metrics['fnr'], 2, True),
        format_avg(baseline_unaware_metrics['fnr'], 2, True),
        format_avg(ablation_metrics['fnr'], 2, True),
        format_avg(fair_metrics['fnr'], 2, True),
        format_avg(fair_unaware_metrics['fnr'], 2, True)
        ],
'FPR (95% CI)': [
        format_avg(baseline_metrics['fpr'], 2, True),
        format_avg(baseline_unaware_metrics['fpr'], 2, True),
        format_avg(ablation_metrics['fpr'], 2, True),
        format_avg(fair_metrics['fpr'], 2, True),
        format_avg(fair_unaware_metrics['fpr'], 2, True)
        ],
'Recall (95% CI)': [
        format_avg(baseline_metrics['recall'], 2, True),
        format_avg(baseline_unaware_metrics['recall'], 2, True),
        format_avg(ablation_metrics['recall'], 2, True),
        format_avg(fair_metrics['recall'], 2, True),
        format_avg(fair_unaware_metrics['recall'], 2, True)
        ],
'PPV (95% CI)': [
        format_avg(baseline_metrics['ppv'], 2, True),
        format_avg(baseline_unaware_metrics['ppv'], 2, True),
        format_avg(ablation_metrics['ppv'], 2, True),
        format_avg(fair_metrics['ppv'], 2, True),
        format_avg(fair_unaware_metrics['ppv'], 2, True)
        ]
})

print(perf_summary.to_markdown(index=False, numalign="right"))

In [ ]:
# Statistical significance of the difference in error rate disparity
def fnr_fpr_disparity(metrics, ttest=True):
  metrics['fpr_diff'] = metrics['fpr_0'] - metrics['fpr_1']
  metrics['fnr_diff'] = metrics['fnr_0'] - metrics['fnr_1']

  if ttest:
    fpr_ttest = ttest_rel(baseline_metrics['fpr_diff'],
                           metrics['fpr_diff'],
                           alternative='two-sided',
                           nan_policy='omit')
    fnr_ttest = ttest_rel(baseline_metrics['fnr_diff'],
                           metrics['fnr_diff'],
                           alternative='two-sided',
                           nan_policy='omit')
    return fpr_ttest, fnr_ttest

fnr_fpr_disparity(baseline_metrics, ttest=False)
fpr_baseline_un_ttest, fnr_baseline_un_ttest = fnr_fpr_disparity(baseline_unaware_metrics)
fpr_ablation_ttest, fnr_ablation_ttest = fnr_fpr_disparity(ablation_metrics)
fpr_fair_ttest, fnr_fair_ttest = fnr_fpr_disparity(fair_metrics)
fpr_fair_un_ttest, fnr_fair_un_ttest = fnr_fpr_disparity(fair_unaware_metrics)

fnr_disparities_summary = pd.DataFrame({
  'Model': [
    'Baseline: Xdesc, Xcorr, Xind, Sens', 
    'Baseline S-unaware: Xdesc, Xcorr, Xind',
    'Ablation: Xcorr, Xind',
    'Fair: Udesc, Xcorr, Xind, Sens',
    'Fair S-unaware: Udesc, Xcorr, Xind'],
  'Avg Male FNR %': [
    format_avg(baseline_metrics['fnr_1'], 2, True),
    format_avg(baseline_unaware_metrics['fnr_1'], 2, True),
    format_avg(ablation_metrics['fnr_1'], 2, True),
    format_avg(fair_metrics['fnr_1'], 2, True),
    format_avg(fair_unaware_metrics['fnr_1'], 2, True)],
  'Avg Female FNR %': [
    format_avg(baseline_metrics['fnr_0'], 2, True),
    format_avg(baseline_unaware_metrics['fnr_0'], 2, True),
    format_avg(ablation_metrics['fnr_0'], 2, True),
    format_avg(fair_metrics['fnr_0'], 2, True),
    format_avg(fair_unaware_metrics['fnr_0'], 2, True)],
  'Avg FNR Disparity %': [
    format_avg(baseline_metrics['fnr_diff'], 2, True),
    format_avg(baseline_unaware_metrics['fnr_diff'], 2, True),
    format_avg(ablation_metrics['fnr_diff'], 2, True),
    format_avg(fair_metrics['fnr_diff'], 2, True),
    format_avg(fair_unaware_metrics['fnr_diff'], 2, True)],
  'Paired t-stat': [
    0,
    round(fnr_baseline_un_ttest.statistic, 2),
    round(fnr_ablation_ttest.statistic, 2),
    round(fnr_fair_ttest.statistic, 2),
    round(fnr_fair_un_ttest.statistic, 2)],
  'p-value': [
    "",
    "< 0.01" if fnr_baseline_un_ttest.pvalue < 0.01 else round(fnr_baseline_un_ttest.pvalue, 3),
    "< 0.01" if fnr_ablation_ttest.pvalue < 0.01 else round(fnr_ablation_ttest.pvalue, 3),
    "< 0.01" if fnr_fair_ttest.pvalue < 0.01 else round(fnr_fair_ttest.pvalue, 6),
    "< 0.01" if fnr_fair_un_ttest.pvalue < 0.01 else round(fnr_fair_un_ttest.pvalue, 6)],
})


print(fnr_disparities_summary.to_markdown(index=False))


In [ ]:
fpr_disparities_summary = pd.DataFrame({
  'Model': [
    'Baseline: Xdesc, Xcorr, Xind, Sens', 
    'Baseline S-unaware: Xdesc, Xcorr, Xind',
    'Ablation: Xcorr, Xind',
    'Fair: Udesc, Xcorr, Xind, Sens',
    'Fair S-unaware: Udesc, Xcorr, Xind'],
  'Avg Male FPR': [
    format_avg(baseline_metrics['fpr_1'], 2, True),
    format_avg(baseline_unaware_metrics['fpr_1'], 2, True),
    format_avg(ablation_metrics['fpr_1'], 2, True),
    format_avg(fair_metrics['fpr_1'], 2, True),
    format_avg(fair_unaware_metrics['fpr_1'], 2, True)],
  'Avg Female FPR': [
    format_avg(baseline_metrics['fpr_0'], 2, True),
    format_avg(baseline_unaware_metrics['fpr_0'], 2, True),
    format_avg(ablation_metrics['fpr_0'], 2, True),
    format_avg(fair_metrics['fpr_0'], 2, True),
    format_avg(fair_unaware_metrics['fpr_0'], 2, True)],
  'Avg FPR Disparity': [
    format_avg(baseline_metrics['fpr_diff'], 2, True),
    format_avg(baseline_unaware_metrics['fpr_diff'], 2, True),
    format_avg(ablation_metrics['fpr_diff'], 2, True),
    format_avg(fair_metrics['fpr_diff'], 2, True),
    format_avg(fair_unaware_metrics['fpr_diff'], 2, True)],
  'Paired t-stat': [
    0,
    round(fpr_baseline_un_ttest.statistic, 2),
    round(fpr_ablation_ttest.statistic, 2),
    round(fpr_fair_ttest.statistic, 2),
    round(fpr_fair_un_ttest.statistic, 2)],
  'p-value': [
    "",
    "< 0.01" if fpr_baseline_un_ttest.pvalue < 0.01 else round(fpr_baseline_un_ttest.pvalue, 6),
    "< 0.01" if fpr_ablation_ttest.pvalue < 0.01 else round(fpr_ablation_ttest.pvalue, 6),
    "< 0.01" if fpr_fair_ttest.pvalue < 0.01 else round(fpr_fair_ttest.pvalue, 6),
    "< 0.01" if fpr_fair_un_ttest.pvalue < 0.01 else round(fpr_fair_un_ttest.pvalue, 6)],
})


print(fpr_disparities_summary.to_markdown(index=False))

In [ ]:
# Visualising difference in FNR disparity between baseline and fair model
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(4, 4))
sns.scatterplot(baseline_metrics, x='fnr_1', y='fnr_0', ax=ax, c='#1f77b4')
sns.scatterplot(fair_metrics, x='fnr_1', y='fnr_0', ax=ax, c='#ff7f0e')
plt.plot([0,.7],[0,.7],c='g')

fair_vx = (fair_metrics['fnr_1'] - baseline_metrics['fnr_1']).sum(axis=0) / baseline_metrics.shape[0]
fair_vy = (fair_metrics['fnr_0'] - baseline_metrics['fnr_0']).sum(axis=0) / baseline_metrics.shape[0]
fair_average_diff = mpatches.Arrow(0.05, 0.1, fair_vx, fair_vy, width=0.05, color="#ff7f0e" )
ax.add_patch(fair_average_diff)

fig.suptitle('False Negative Rate by subgroup')
plt.xlabel('Male FNR')
plt.ylabel('Female FNR')
plt.legend(labels=['Baseline model', 'Fair model',
                   'Equal error rates between subgroups',
                   'Average change in FNR from Baseline to Fair model'],
           loc='upper left', bbox_to_anchor=(1, 1))
# ax.set_xlim([0,0.3])
# ax.set_ylim([-.05,0.7])
ax.set_aspect('equal')
plt.show()

In [ ]:
# Visualising difference in FNR disparity between baseline and fair model 2
import matplotlib.patches as mpatches

fig, ax = plt.subplots(figsize=(4, 4))
sns.scatterplot(baseline_metrics, x='fnr_1', y='fnr_0', ax=ax, c='#1f77b4')
sns.scatterplot(baseline_unaware_metrics, x='fnr_1', y='fnr_0', ax=ax, c='#ff7f0e')
plt.plot([0,.7],[0,.7],c='g')

fair_vx = (baseline_unaware_metrics['fnr_1'] - baseline_metrics['fnr_1']).sum(axis=0) / baseline_metrics.shape[0]
fair_vy = (baseline_unaware_metrics['fnr_0'] - baseline_metrics['fnr_0']).sum(axis=0) / baseline_metrics.shape[0]
fair_average_diff = mpatches.Arrow(0.01, 0.1, fair_vx, fair_vy, width=0.05, color="#ff7f0e" )
ax.add_patch(fair_average_diff)

fig.suptitle('False Negative Rate by subgroup')
plt.xlabel('Male FNR')
plt.ylabel('Female FNR')
plt.legend(labels=['Baseline model', 'Baseline S-unaware model',
                   'Equal error rates between subgroups',
                   'Average change in FNR from Baseline to Baseline S-unaware model'],
           loc='upper left', bbox_to_anchor=(1, 1))
# ax.set_xlim([0,0.3])
# ax.set_ylim([-.05,0.7])
ax.set_aspect('equal')
plt.show()

# Feature importances

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(6, 8), sharex=True)

sns.barplot(baseline_feat_imp, x="importance_mean", y="feature", orient="h", ax=axes[0], width=0.5)
sns.barplot(fair_unaware_feat_imp, x="importance_mean", y="feature", orient="h", ax=axes[1], width=0.5)

axes[0].set_xlabel("")
axes[0].set_ylabel("")
axes[0].set_title("Baseline model: feature importances")
axes[1].set_xlabel("")
axes[1].set_ylabel("")
axes[1].set_title("Fair S-Unaware model: feature importances")

plt.show()